# Training & Evaluating the model

## 0. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
import joblib

ARTIFACT_PATH = r"C:\Users\User\Documents\Key Future Docs\Projects\Phising URL\dataset\artifact.joblib"
RANDOM_STATE = 42

## 1. Load & Prepare the data for training

### A. Load artifact

In [ ]:
artifact = joblib.load("artifacts/features.joblib")
train_df = artifact["train_df"]
test_df = artifact["test_df"]
train_ngrams = artifact["train_ngrams"]
test_ngrams = artifact["test_ngrams"]
vectorizer = artifact["vectorizer"]
vectorizer_cols = artifact["vectorizer_cols"]

print(f"Train df shape: {train_df.shape}, Test df shape: {test_df.shape}")
print(f"Train ngrams shape: {train_ngrams.shape}, Test ngrams shape: {test_ngrams.shape}")
train_df.head()

### B. Compress n-grams with Single Vector Decomposition (SVD)

In [ ]:
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
X_ngram_train = svd.fit_transform(train_ngrams)
X_ngram_test = svd.transform(test_ngrams)

### C. Build stack for training

In [ ]:
X_train = train_df.drop(columns=['url', 'is_phishing']).to_numpy()
X_test = test_df.drop(columns=['url', 'is_phishing']).to_numpy()

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train = hstack([X_train, X_ngram_train])
y_train = train_df['is_phishing']

X_test = hstack([X_test, X_ngram_test])
y_test = test_df['is_phishing']

## 2. Train an MLP

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=RANDOM_STATE,
    early_stopping=True,
    random_state=RANDOM_STATE
)

mlp.fit(X_train, y_train)

In [ ]:
preds = mlp.predict(X_test)
probs = mlp.predict_proba(X_test)

accuracy = accuracy_score(y_test, preds)
auc = roc_auc_score(y_test, probs)

## 3. Evalutate the model

In [ ]:
print("Classification Report for MLP:")
print(classification_report(y_test, preds))

print("Confusion Matrix:")
print(confusion_matrix(y_test, preds))